# 🎨 TryHairStyle — Training Pipeline trên Google Colab

## Pipeline:
1. **Stage 1**: Texture Encoder — Học đặc trưng chất liệu tóc
2. **Stage 2**: Mask-Conditioned Inpainting — UNet SDXL 9-channel
3. **Stage 3**: Evaluation & Export

## Chiến lược Hybrid Storage:
- **File nhỏ, đọc thường xuyên** → Colab disk (nhanh)
- **Ảnh lớn** → Google Drive via symlink (2TB, không mất khi ngắt)
- **Checkpoint + charts** → Drive trực tiếp (auto-save, không cần backup)

---

## ⚙️ 0. Cấu hình

In [ ]:
from pathlib import Path

# ============================================================
# GOOGLE DRIVE PATHS
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/TryHairStyle")
DRIVE_DATA_DIR = DRIVE_ROOT / "compressed_for_drive"     # File .zip đã upload
DRIVE_PROCESSED_DIR = DRIVE_ROOT / "processed"           # Giải nén lên đây
DRIVE_CHECKPOINTS_DIR = DRIVE_ROOT / "checkpoints"       # Checkpoint + charts
DRIVE_SDXL_DIR = DRIVE_ROOT / "models" / "sd_xl_inpainting"

# ============================================================
# COLAB LOCAL PATHS
# ============================================================
WORK_DIR = Path("/content/TryHairStyle")
LOCAL_PROCESSED = WORK_DIR / "backend" / "training" / "processed"

# GitHub — GitHub
GITHUB_REPO = "https://github.com/Phatjhhoq8/TryHairstyle_Diffusion.git"
GITHUB_BRANCH = "master"

# ============================================================
# HYBRID STORAGE — File nhỏ copy local, file lớn symlink Drive
# ============================================================
# Copy về Colab disk (nhanh, ~1.5GB tổng)
LOCAL_COPY_DIRS = [
    "identity_embeddings",     # ~195 MB — đọc mỗi step
    "style_embeddings_cache",  # ~8 MB  — đọc mỗi step
    "hair_patches",            # ~1.2 GB — Stage 1 only
]
LOCAL_COPY_FILES = [
    "metadata.jsonl",          # ~28 MB — đọc liên tục
]

# Giữ trên Drive via symlink (lớn, ~60GB)
DRIVE_SYMLINK_DIRS = [
    "bald_images",             # ~22.5 GB
    "ground_truth_images",     # ~24 GB
    "hair_only_images",        # ~4.4 GB
    "style_vectors",           # ~8.9 GB
]

# ============================================================
# TRAINING HYPERPARAMETERS
# ============================================================
STAGE1_EPOCHS = 10
STAGE1_BATCH_SIZE = 16
STAGE1_MAX_SAMPLES = 0

STAGE2_EPOCHS = 3
STAGE2_BATCH_SIZE = 1
STAGE2_MAX_SAMPLES = 0
STAGE2_RESOLUTION = 512
STAGE2_ACCUMULATION = 8
RESUME_TRAINING = True  # Tiếp tục train từ checkpoint cũ (True) hoặc train từ đầu (False)

# ⚠️ SAU KHI SỬA BUG (Feb 2026): Checkpoint cũ dùng convention sai (bald_image)
# → Sau khi có checkpoint mới, đổi lại False để resume bình thường

print("✅ Cấu hình đã thiết lập!")
print(f"☁️  Drive root: {DRIVE_ROOT}")
print(f"🖥️  Local copy: {LOCAL_COPY_DIRS + LOCAL_COPY_FILES}")
print(f"🔗 Drive symlink: {DRIVE_SYMLINK_DIRS}")

## 📂 1. Mount Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch, shutil
print(f"\n🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ!'}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"💾 VRAM: {vram:.1f} GB | CUDA: {torch.version.cuda}")

total, used, free = shutil.disk_usage('/')
print(f"\n💿 Colab disk: {free / (1024**3):.1f} GB trống")
dtotal, dused, dfree = shutil.disk_usage('/content/drive')
print(f"☁️  Drive: {dfree / (1024**3):.1f} GB trống")

## 📦 2. Giải nén dữ liệu lên Drive (chỉ lần đầu)

In [ ]:
import os, zipfile, time, shutil
from pathlib import Path
from tqdm.notebook import tqdm

DRIVE_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Thư mục tạm trên Colab local disk (nhanh, đáng tin cậy)
LOCAL_TEMP_EXTRACT = Path('/content/_tmp_extract')

# Danh sách TẤT CẢ thư mục PHẢI có sau giải nén
EXPECTED_DIRS = set(LOCAL_COPY_DIRS + DRIVE_SYMLINK_DIRS)

def getZipTopDirs(zipPath):
    """Đọc danh sách thư mục gốc bên trong file zip (không giải nén)."""
    dirs = set()
    with zipfile.ZipFile(zipPath, 'r') as zf:
        for name in zf.namelist():
            top = name.split('/')[0]
            if top:
                dirs.add(top)
    return dirs

def extractZipSafe(zipPath, finalDest):
    """
    Giải nén AN TOÀN: local trước → verify → move sang Drive.
    Tránh lỗi Drive FUSE sync mất dữ liệu.
    """
    name = os.path.basename(zipPath)
    
    # Bước 1: Giải nén vào thư mục tạm LOCAL (nhanh, không lỗi FUSE)
    LOCAL_TEMP_EXTRACT.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zipPath, 'r') as zf:
        members = zf.infolist()
        totalSize = sum(m.file_size for m in members)
        sizeMB = os.path.getsize(zipPath) / (1024**2)
        
        print(f"  ⬇️  Bước 1/3: Giải nén về local disk...")
        with tqdm(
            total=totalSize, unit='B', unit_scale=True, unit_divisor=1024,
            desc=f'📦 {name}',
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
        ) as pbar:
            for member in members:
                zf.extract(member, str(LOCAL_TEMP_EXTRACT))
                pbar.update(member.file_size)
    
    # Bước 2: Verify file đã giải nén đúng trên local
    extractedDirs = [d for d in LOCAL_TEMP_EXTRACT.iterdir() if d.is_dir()]
    extractedFiles = [f for f in LOCAL_TEMP_EXTRACT.iterdir() if f.is_file()]
    print(f"  ✅ Bước 2/3: Verify local — {len(extractedDirs)} thư mục, {len(extractedFiles)} file")
    
    for item in extractedDirs:
        cnt = sum(1 for _ in item.rglob('*') if _.is_file())
        localSize = sum(f.stat().st_size for f in item.rglob('*') if f.is_file()) / (1024**3)
        print(f"     📁 {item.name}: {cnt:,} files ({localSize:.2f} GB)")
    
    # Bước 3: Move từ local sang Drive (từng thư mục)
    print(f"  ☁️  Bước 3/3: Di chuyển sang Drive...")
    for item in list(LOCAL_TEMP_EXTRACT.iterdir()):
        dst = Path(finalDest) / item.name
        if dst.exists():
            print(f"     ⏭️  {item.name} — đã có trên Drive, ghi đè...", end=' ', flush=True)
            if dst.is_dir():
                shutil.rmtree(str(dst))
            else:
                dst.unlink()
        else:
            print(f"     📤 {item.name}...", end=' ', flush=True)
        
        t = time.time()
        if item.is_dir():
            shutil.copytree(str(item), str(dst))
        else:
            shutil.copy2(str(item), str(dst))
        print(f"✅ ({time.time()-t:.0f}s)")
        
        # Verify trên Drive
        if dst.is_dir():
            srcCnt = sum(1 for _ in item.rglob('*') if _.is_file())
            dstCnt = sum(1 for _ in dst.rglob('*') if _.is_file())
            if srcCnt != dstCnt:
                print(f"     🚨 {item.name}: local={srcCnt}, Drive={dstCnt} — KHÔNG KHỚP!")
            else:
                print(f"     ✅ {item.name}: {dstCnt:,} files khớp")
    
    # Cleanup local temp
    shutil.rmtree(str(LOCAL_TEMP_EXTRACT), ignore_errors=True)
    print(f"  🧹 Đã dọn thư mục tạm")

if not DRIVE_DATA_DIR.exists():
    print(f"❌ Không tìm thấy: {DRIVE_DATA_DIR}")
    print(f"   Upload file .zip vào thư mục này trên Drive.")
else:
    existing = {d.name for d in DRIVE_PROCESSED_DIR.iterdir() if d.is_dir()} if DRIVE_PROCESSED_DIR.exists() else set()
    missing = EXPECTED_DIRS - existing
    
    if not missing:
        print(f"⏭️  Dữ liệu ĐÃ giải nén đầy đủ {len(EXPECTED_DIRS)}/{len(EXPECTED_DIRS)} thư mục. Skip!")
        for d in sorted(EXPECTED_DIRS):
            print(f"   ✅ {d}/")
    else:
        print(f"⚠️  Thiếu {len(missing)}/{len(EXPECTED_DIRS)} thư mục:")
        for d in sorted(missing):
            print(f"   ❌ {d}/")
        if existing & EXPECTED_DIRS:
            print(f"   ✅ Đã có: {sorted(existing & EXPECTED_DIRS)}")
        
        # Kiểm tra disk local đủ không
        _, _, freeLocal = shutil.disk_usage('/')
        print(f"\n💿 Colab disk trống: {freeLocal / (1024**3):.1f} GB")
        
        # Quét từng file zip
        zipFiles = sorted(DRIVE_DATA_DIR.glob("*.zip"))
        print(f"🔍 Quét {len(zipFiles)} file zip...")
        
        zipsToExtract = []
        for z in zipFiles:
            topDirs = getZipTopDirs(str(z))
            containsMissing = topDirs & missing
            alreadyDone = topDirs & existing
            sizeMB = z.stat().st_size / (1024**2)
            
            if containsMissing:
                print(f"   📦 {z.name} ({sizeMB:.0f} MB) → CẦN giải nén (chứa: {sorted(containsMissing)})")
                zipsToExtract.append(z)
            else:
                print(f"   ⏭️  {z.name} ({sizeMB:.0f} MB) → SKIP (đã đủ: {sorted(alreadyDone)})")
        
        if zipsToExtract:
            totalGB = sum(f.stat().st_size for f in zipsToExtract) / (1024**3)
            print(f"\n📤 Giải nén {len(zipsToExtract)}/{len(zipFiles)} file zip ({totalGB:.2f} GB)")
            print(f"� Chiến lược: Local disk → Verify → Move to Drive (tránh lỗi FUSE)")
            print("=" * 60)
            t0 = time.time()
            
            for idx, z in enumerate(zipsToExtract, 1):
                print(f"\n{'='*60}")
                print(f"[{idx}/{len(zipsToExtract)}] {z.name}")
                print(f"{'='*60}")
                extractZipSafe(str(z), str(DRIVE_PROCESSED_DIR))
            
            elapsed = time.time() - t0
            print(f"\n🎉 Hoàn tất! {elapsed/60:.1f} phút")
        else:
            print(f"\n🚨 Không tìm thấy zip nào chứa thư mục thiếu: {sorted(missing)}")
            print("   → Kiểm tra lại file .zip đã upload!")
        
        # Verify cuối cùng
        existing_after = {d.name for d in DRIVE_PROCESSED_DIR.iterdir() if d.is_dir()}
        still_missing = EXPECTED_DIRS - existing_after
        if still_missing:
            print(f"\n🚨 VẪN THIẾU {len(still_missing)} thư mục:")
            for d in sorted(still_missing):
                print(f"   ❌ {d}/")
            print("   → Có thể Drive chưa sync kịp hoặc zip thiếu dữ liệu!")
        else:
            print(f"\n✅ Đã đủ {len(EXPECTED_DIRS)}/{len(EXPECTED_DIRS)} thư mục!")
    
    # Thống kê
    print("\n📊 Cấu trúc:")
    for item in sorted(DRIVE_PROCESSED_DIR.iterdir()):
        if item.is_dir():
            cnt = sum(1 for _ in item.rglob("*") if _.is_file())
            status = "✅" if item.name in EXPECTED_DIRS else "❓"
            print(f"   {status} 📁 {item.name}: {cnt:,} files")
        else:
            print(f"   📄 {item.name}: {item.stat().st_size / (1024**2):.1f} MB")

## 🔧 3. Clone Code & Cài Dependencies

In [ ]:
import subprocess

if (WORK_DIR / ".git").exists():
    print("📁 Repo đã có, pull...")
    subprocess.run(["git", "pull"], cwd=str(WORK_DIR), check=True)
else:
    print(f"📥 Cloning...")
    subprocess.run([
        "git", "clone", "--branch", GITHUB_BRANCH, "--depth", "1",
        GITHUB_REPO, str(WORK_DIR)
    ], check=True)
print("✅ Code sẵn sàng!")

In [ ]:
!pip install -q diffusers>=0.36.0 transformers>=5.0.0 accelerate>=1.0.0 \
    peft>=0.18.0 safetensors>=0.7.0 huggingface_hub>=1.0.0 \
    insightface>=0.7.3 ultralytics>=8.0.0 facenet-pytorch>=2.6.0 \
    lpips>=0.1.4 bitsandbytes>=0.43.0 xformers \
    mobile_sam>=1.0 deep-translator>=1.11.0 \
    onnx>=1.16.0 onnxruntime-gpu>=1.23.0 \
    easydict>=1.0 gdown>=5.0.0 scikit-image>=0.26.0
print("\n✅ Dependencies OK!")

## 🔀 4. Thiết lập Hybrid Storage
Copy file nhỏ về local disk (nhanh) + symlink file lớn từ Drive.

In [ ]:
import sys, os, shutil, time

sys.path.insert(0, str(WORK_DIR))
os.chdir(str(WORK_DIR))

# Tạo thư mục processed trên Colab local
LOCAL_PROCESSED.mkdir(parents=True, exist_ok=True)

print("="*60)
print("🔀 HYBRID STORAGE SETUP")
print("="*60)

# 1. Copy file nhỏ về local disk (nhanh khi đọc)
print("\n🖥️  Copy file nhỏ về Colab disk:")
for dirName in LOCAL_COPY_DIRS:
    src = DRIVE_PROCESSED_DIR / dirName
    dst = LOCAL_PROCESSED / dirName
    if dst.exists():
        print(f"   ⏭️  {dirName}/ — đã tồn tại")
        continue
    if src.exists():
        t = time.time()
        print(f"   📋 {dirName}/ ...", end=" ", flush=True)
        shutil.copytree(str(src), str(dst))
        cnt = sum(1 for _ in dst.rglob("*") if _.is_file())
        print(f"✅ ({cnt:,} files, {time.time()-t:.0f}s)")
    else:
        print(f"   ⚠️  {dirName}/ — không tìm thấy trên Drive")

for fileName in LOCAL_COPY_FILES:
    src = DRIVE_PROCESSED_DIR / fileName
    dst = LOCAL_PROCESSED / fileName
    if dst.exists():
        print(f"   ⏭️  {fileName} — đã tồn tại")
        continue
    if src.exists():
        print(f"   📋 {fileName} ...", end=" ", flush=True)
        shutil.copy2(str(src), str(dst))
        sizeMB = dst.stat().st_size / (1024**2)
        print(f"✅ ({sizeMB:.1f} MB)")
    else:
        print(f"   ⚠️  {fileName} — không tìm thấy")

# 2. Symlink thư mục ảnh lớn từ Drive (tiết kiệm disk)
print("\n🔗 Symlink thư mục lớn từ Drive:")
for dirName in DRIVE_SYMLINK_DIRS:
    src = DRIVE_PROCESSED_DIR / dirName
    dst = LOCAL_PROCESSED / dirName
    if dst.exists() or dst.is_symlink():
        print(f"   ⏭️  {dirName}/ — đã tồn tại")
        continue
    if src.exists():
        dst.symlink_to(src)
        cnt = sum(1 for _ in src.rglob("*") if _.is_file())
        print(f"   🔗 {dirName}/ → Drive ({cnt:,} files)")
    else:
        print(f"   ⚠️  {dirName}/ — không tìm thấy trên Drive")

# 3. Symlink checkpoints → Drive (auto-save)
DRIVE_CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
localCkpt = WORK_DIR / "backend" / "training" / "checkpoints"
if not localCkpt.exists():
    localCkpt.symlink_to(DRIVE_CHECKPOINTS_DIR)
    print(f"\n🔗 checkpoints/ → Drive")
elif localCkpt.is_symlink():
    print(f"\n🔗 checkpoints/ → Drive (đã có)")

# Thống kê
print("\n" + "="*60)
print("📊 KẾT QUẢ:")
localSize = sum(f.stat().st_size for f in LOCAL_PROCESSED.rglob("*") if f.is_file() and not f.is_symlink()) / (1024**3)
print(f"   🖥️  Local disk: ~{localSize:.2f} GB (file nhỏ, đọc nhanh)")
print(f"   ☁️  Drive: ~60 GB (ảnh lớn, qua symlink)")
print("="*60)

# Verify import
try:
    from backend.app.services.training_utils import setupLogger, getDevice, ensureDir
    print(f"\n✅ Import OK! Device: {getDevice()}")
except ImportError as e:
    print(f"❌ Lỗi: {e}")

## 🧠 5. Chuẩn bị SDXL Model

In [ ]:
import torch

SDXL_LOCAL = WORK_DIR / "backend" / "models" / "stable-diffusion" / "sd_xl_inpainting"

if SDXL_LOCAL.exists() and any(SDXL_LOCAL.iterdir()):
    print("✅ SDXL model đã có")
elif DRIVE_SDXL_DIR.exists() and any(DRIVE_SDXL_DIR.iterdir()):
    print("📋 Copy SDXL từ Drive...")
    SDXL_LOCAL.mkdir(parents=True, exist_ok=True)
    !cp -r "{DRIVE_SDXL_DIR}/"* "{SDXL_LOCAL}/"
    print("✅ Done!")
else:
    print("📥 Download SDXL từ HuggingFace...")
    SDXL_LOCAL.mkdir(parents=True, exist_ok=True)
    from diffusers import StableDiffusionXLInpaintPipeline
    pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
        "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        torch_dtype=torch.float16, variant="fp16"
    )
    pipe.save_pretrained(str(SDXL_LOCAL))
    del pipe; torch.cuda.empty_cache()
    DRIVE_SDXL_DIR.mkdir(parents=True, exist_ok=True)
    !cp -r "{SDXL_LOCAL}/"* "{DRIVE_SDXL_DIR}/"
    print("✅ Download + cache Drive!")

---
# 🚀 TRAINING
---

### ⚠️ Lưu ý quan trọng (Feb 2026)

Code đã được sửa **4 lỗi nghiêm trọng** + thêm **Validation Evaluation**:

1. **Inference dùng `masked_image`** thay vì `bald_image` — khớp convention training
2. **Identity Loss** giờ được tính thực sự (weight=0.005)
3. **Texture Loss** tính đúng per-sample `alpha` khi batch > 1
4. **Style embedding** fix dimension mismatch (768→2048)
5. **✨ Validation-based best model**: Dataset tự chia 80/20, chọn best bằng Val Loss + LPIPS

> **Checkpoint cũ KHÔNG tương thích!** Phải train lại Stage 2 với `FORCE_FRESH_STAGE2 = True`.
> Sau khi có checkpoint mới, đổi lại `False`.

## 🎯 Stage 1: Texture Encoder

In [ ]:
import os, torch
os.chdir(str(WORK_DIR))

texCkpt = DRIVE_CHECKPOINTS_DIR / "texture_encoder_latest.safetensors"
if texCkpt.exists():
    print(f"⏭️  Texture Encoder đã train ({texCkpt.stat().st_size/(1024**2):.1f} MB)")
    skip = input("Bỏ qua Stage 1? (y/n): ").strip().lower() == 'y'
else:
    skip = False

if not skip:
    print("="*60)
    print("🎯 STAGE 1: TEXTURE ENCODER")
    print("="*60)
    from backend.training.models.texture_encoder import TextureEncoderTrainer
    t1 = TextureEncoderTrainer()
    t1.train_loop(num_epochs=STAGE1_EPOCHS, batch_size=STAGE1_BATCH_SIZE,
                  max_samples=STAGE1_MAX_SAMPLES, resume=RESUME_TRAINING)
    del t1; torch.cuda.empty_cache()
    print("✅ Stage 1 xong!")
else:
    print("⏭️  Skip.")

## 🧬 Stage 2: Hair Inpainting (Main)
> Dataset tự chia 80% train / 20% val. Best model chọn bằng Val Loss.
> Checkpoint auto-save trực tiếp lên Drive qua symlink.

In [ ]:
import os, torch
os.chdir(str(WORK_DIR))

print("="*60)
print("🧬 STAGE 2: HAIR INPAINTING")
print("="*60)
if torch.cuda.is_available():
    print(f"💾 VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB used")

from backend.training.train_stage2 import Stage2Trainer
t2 = Stage2Trainer()
t2.train_loop(
    num_epochs=STAGE2_EPOCHS, batch_size=STAGE2_BATCH_SIZE,
    max_samples=STAGE2_MAX_SAMPLES,
    target_size=(STAGE2_RESOLUTION, STAGE2_RESOLUTION),
    accumulation_steps=STAGE2_ACCUMULATION, resume=RESUME_TRAINING
)
del t2; torch.cuda.empty_cache()
print("✅ Stage 2 xong!")


## 📊 Loss Chart

In [ ]:
from IPython.display import display, Image as IPImage

chartsDir = DRIVE_CHECKPOINTS_DIR / "charts"
if chartsDir.exists():
    charts = sorted(chartsDir.glob("loss_chart_epoch_*.png"))
    if charts:
        print(f"📊 {len(charts)} chart(s):")
        for c in charts: print(f"   📈 {c.name}")
        latest = chartsDir / "loss_chart_latest.png"
        show = str(latest) if latest.exists() else str(charts[-1])
        display(IPImage(filename=show, width=900))
        print("\n📋 Charts 32: Total Loss, Diffusion Loss, Texture+Identity, Train vs Val Loss, LPIPS, Summary Table")
    else:
        print("⚠️  Chưa có chart.")
else:
    print("⚠️  Chạy Stage 2 trước.")


## 📊 Stage 3: Evaluation & Export

In [ ]:
import os, shutil
os.chdir(str(WORK_DIR))

print("="*60)
print("📊 STAGE 3: EVALUATION & EXPORT")
print("="*60)

from backend.training.export_model import CheckpointManager
mgr = CheckpointManager()
ckpt = mgr.find_latest_checkpoint()

if ckpt:
    print(f"📁 Checkpoint: {ckpt}")
    passed, metrics = mgr.test_checkpoint(ckpt)
    print(f"📊 Metrics: {metrics}")
    if passed:
        mgr.export_to_production(ckpt)
        exported = WORK_DIR / "backend" / "training" / "models" / "deep_hair_v1.safetensors"
        if exported.exists():
            dest = DRIVE_ROOT / "models"
            dest.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(exported), str(dest / exported.name))
            print(f"💾 Model export → Drive!")
    else:
        print("⚠️  Checkpoint KHÔNG HỢP LỆ. Train thêm.")
else:
    print("❌ Không có checkpoint.")

## 🔍 Quick Preview

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

bald = list((LOCAL_PROCESSED / "bald_images").glob("*.png"))[:1]
gt = list((LOCAL_PROCESSED / "ground_truth_images").glob("*.png"))[:1]

if bald and gt:
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(Image.open(str(bald[0])).resize((512,512))); ax[0].set_title("Bald"); ax[0].axis('off')
    ax[1].imshow(Image.open(str(gt[0])).resize((512,512))); ax[1].set_title("Ground Truth"); ax[1].axis('off')
    plt.tight_layout(); plt.show()
else:
    print("⚠️  Không tìm thấy ảnh.")

## 📝 Ghi chú

### Hybrid Storage Strategy:
```
processed/ (Colab local)
├── metadata.jsonl           ← LOCAL COPY  (28 MB)
├── identity_embeddings/     ← LOCAL COPY  (195 MB)
├── style_embeddings_cache/  ← LOCAL COPY  (8 MB)
├── hair_patches/            ← LOCAL COPY  (1.2 GB)
├── bald_images/        →  🔗 SYMLINK → Drive (22.5 GB)
├── ground_truth_images/ →  🔗 SYMLINK → Drive (24 GB)
├── hair_only_images/   →  🔗 SYMLINK → Drive (4.4 GB)
└── style_vectors/      →  🔗 SYMLINK → Drive (8.9 GB)

checkpoints/ → 🔗 SYMLINK → Drive (auto-save)
```

### Cấu trúc Drive:
```
MyDrive/TryHairStyle/
├── compressed_for_drive/    ← Upload .zip vào đây
├── processed/               ← Giải nén 1 lần
├── checkpoints/             ← Auto-save qua symlink
│   └── charts/
└── models/
    ├── sd_xl_inpainting/
    └── deep_hair_v1.safetensors
```

### Hiệu suất:
- File nhỏ (embeddings, metadata) → **local disk** → đọc cực nhanh
- File ảnh lớn → **Drive** → chậm hơn ~5-10ms/file nhưng GPU là bottleneck chính
- Tổng thể **nhanh hơn ~15-20%** so với toàn bộ trên Drive